# Flickr8k Image-Text ImageBind-Style Mini

This Kaggle notebook adapts the existing ESC-50 ImageBind-style idea from audio-text to image-text on Flickr8k.

Goal: train a small model from scratch for 5 epochs using `(image, caption)` positive pairs, learn a joint embedding space with symmetric InfoNCE contrastive loss, and evaluate image-to-text plus text-to-image retrieval.

This notebook does not use pretrained CLIP, OpenCLIP, ImageBind, or audio. The image encoder is a randomly initialized ResNet18 and the text encoder is a small trainable embedding + GRU network.

## 1. Imports and Config

The config defaults are Kaggle-friendly. `FORCE_CPU=False` allows GPU use when CUDA is actually compatible; the next cell tests CUDA with a small matrix multiplication and falls back to CPU if needed.

In [ ]:
import json
import math
import os
import random
import re
import string
import textwrap
import time
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms
from torchvision.models import resnet18

try:
    from sklearn.metrics import pairwise_distances
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/flickr8k"
    IMAGE_SIZE: int = 224
    EMBED_DIM: int = 256
    BATCH_SIZE: int = 32
    EPOCHS: int = 5
    LR: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    TEMPERATURE: float = 0.07
    NUM_WORKERS: int = 2
    FORCE_CPU: bool = False
    MAX_LEN: int = 32
    OUTPUT_DIR: str = "outputs"
    CHECKPOINT_DIR: str = "checkpoints"
    ABLATION_MAX_BATCHES: int = 100

CFG = Config()
OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
CHECKPOINT_DIR = Path(CFG.CHECKPOINT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(asdict(CFG))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)

## 2. Device Setup With CUDA Compatibility Test

Some Kaggle images can expose a CUDA device that is incompatible with the installed PyTorch build. This cell tests CUDA with a small matrix multiplication first. If it catches a CUDA runtime error such as `no kernel image is available for execution on the device`, training continues on CPU.

In [ ]:
def setup_device(force_cpu=False):
    if force_cpu:
        print("FORCE_CPU=True, using CPU.")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        print("CUDA is not available, using CPU.")
        return torch.device("cpu")

    try:
        device = torch.device("cuda")
        a = torch.randn(64, 64, device=device)
        b = torch.randn(64, 64, device=device)
        c = a @ b
        torch.cuda.synchronize()
        print("CUDA test passed:", torch.cuda.get_device_name(0))
        return device
    except RuntimeError as exc:
        msg = str(exc)
        print("CUDA test failed, falling back to CPU.")
        print("RuntimeError:", msg[:500])
        if "no kernel image is available for execution on the device" in msg:
            print("Detected CUDA architecture mismatch: no kernel image is available for execution on the device.")
        return torch.device("cpu")

DEVICE = setup_device(CFG.FORCE_CPU)
print("Device:", DEVICE)

## 3. Flickr8k Path Detection and Caption Parser

The notebook first checks `/kaggle/input/flickr8k`, then common Kaggle alternatives. It recursively detects `captions.txt`, handles both common caption formats, and locates the image directory by checking real image filenames.

In [ ]:
COMMON_DATA_ROOTS = [
    Path(CFG.DATA_ROOT),
    Path("/kaggle/input/flickr8kimagescaptions"),
    Path("/kaggle/input/flickr-8k"),
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def candidate_roots():
    roots = []
    for p in COMMON_DATA_ROOTS:
        if p.exists() and p not in roots:
            roots.append(p)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for child in sorted(kaggle_input.iterdir()):
            if child.is_dir() and child not in roots:
                roots.append(child)
    if not roots:
        roots.append(Path.cwd())
    return roots


def find_captions_file():
    for root in candidate_roots():
        direct = root / "captions.txt"
        if direct.exists():
            return direct
        try:
            matches = list(root.rglob("captions.txt"))
        except Exception as exc:
            print(f"Skipping {root}: {exc}")
            matches = []
        if matches:
            return matches[0]
    searched = ", ".join(str(x) for x in candidate_roots())
    raise FileNotFoundError(f"Could not find captions.txt. Searched: {searched}")


def clean_image_name(name):
    name = str(name).strip().strip('"').strip("'")
    name = re.sub(r"#\d+$", "", name)
    return Path(name).name


def parse_caption_line(line):
    line = line.strip()
    if not line:
        return None
    lower = line.lower()
    if lower in {"image,caption", "image_name,comment", "filename,caption"}:
        return None

    # Newer Flickr8k Kaggle format: image,caption
    if "," in line:
        first, rest = line.split(",", 1)
        if Path(clean_image_name(first)).suffix.lower() in IMAGE_EXTS:
            return clean_image_name(first), rest.strip().strip('"')

    # Older format: image_name#0<TAB>caption or image_name#0 caption
    match = re.match(r"^(.+?\.(?:jpg|jpeg|png|bmp|webp))(?:#\d+)?[\t ]+(.*)$", line, flags=re.IGNORECASE)
    if match:
        return clean_image_name(match.group(1)), match.group(2).strip().strip('"')
    return None


def load_captions(captions_path):
    rows = []
    with open(captions_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parsed = parse_caption_line(line)
            if parsed is not None:
                image, caption = parsed
                if image and caption:
                    rows.append({"image_id": image, "caption": caption})

    if not rows:
        # Fallback for quoted CSV files.
        df = pd.read_csv(captions_path)
        cols = {c.lower().strip(): c for c in df.columns}
        image_col = cols.get("image") or cols.get("filename") or cols.get("image_name")
        caption_col = cols.get("caption") or cols.get("comment") or cols.get("caption_text")
        if image_col is None or caption_col is None:
            raise ValueError(f"Could not parse captions file columns: {df.columns.tolist()}")
        rows = [
            {"image_id": clean_image_name(img), "caption": str(cap).strip()}
            for img, cap in zip(df[image_col], df[caption_col])
            if str(img).strip() and str(cap).strip()
        ]

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    df = df[df["image_id"].str.lower().str.endswith(tuple(IMAGE_EXTS))].reset_index(drop=True)
    return df


def find_images_dir(root, captions_path, image_ids):
    parents = [captions_path.parent, root, root / "flickr8k", root / "Flickr8k"]
    names = ["Images", "images", "Flicker8k_Dataset", "Flickr8k_Dataset", "flickr8k/Images"]
    candidates = []
    for parent in parents:
        for name in names:
            candidates.append(parent / name)
        candidates.append(parent)
    sample_names = list(dict.fromkeys(image_ids[:50]))
    for cand in candidates:
        if cand.exists() and cand.is_dir():
            hits = sum((cand / name).exists() for name in sample_names)
            if hits > 0:
                return cand

    search_base = root if root.exists() else captions_path.parent
    for name in sample_names[:10]:
        try:
            matches = list(search_base.rglob(name))
        except Exception:
            matches = []
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not locate Flickr8k Images directory from captions filenames.")

CAPTIONS_PATH = find_captions_file()
DATA_ROOT = next((r for r in candidate_roots() if CAPTIONS_PATH.is_relative_to(r)), CAPTIONS_PATH.parent)
all_captions_df = load_captions(CAPTIONS_PATH)
IMAGES_DIR = find_images_dir(DATA_ROOT, CAPTIONS_PATH, all_captions_df["image_id"].tolist())

# Keep only rows whose image file exists.
all_captions_df["image_path"] = all_captions_df["image_id"].apply(lambda x: str(IMAGES_DIR / x))
all_captions_df = all_captions_df[all_captions_df["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print("DATA_ROOT:", DATA_ROOT)
print("CAPTIONS_PATH:", CAPTIONS_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("Caption pairs:", len(all_captions_df))
print("Unique images:", all_captions_df["image_id"].nunique())
display(all_captions_df.head())

## 4. Split by Image Filename

Flickr8k has around five captions per image. Splitting by caption would leak the same image into train and test. This split is done by unique `image_id`: 80% train, 10% validation, 10% test.

In [ ]:
def split_by_image_id(df, seed=42, train_ratio=0.8, val_ratio=0.1):
    image_ids = sorted(df["image_id"].unique().tolist())
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    n = len(image_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_ids = set(image_ids[:n_train])
    val_ids = set(image_ids[n_train:n_train + n_val])
    test_ids = set(image_ids[n_train + n_val:])

    train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
    val_df = df[df["image_id"].isin(val_ids)].reset_index(drop=True)
    test_df = df[df["image_id"].isin(test_ids)].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = split_by_image_id(all_captions_df, seed=SEED)
print(f"Train pairs: {len(train_df):,} | images: {train_df['image_id'].nunique():,}")
print(f"Val pairs:   {len(val_df):,} | images: {val_df['image_id'].nunique():,}")
print(f"Test pairs:  {len(test_df):,} | images: {test_df['image_id'].nunique():,}")

assert set(train_df.image_id).isdisjoint(set(val_df.image_id))
assert set(train_df.image_id).isdisjoint(set(test_df.image_id))
assert set(val_df.image_id).isdisjoint(set(test_df.image_id))

## 5. Text Tokenizer

The tokenizer is intentionally simple: lowercase, strip punctuation, build vocabulary from training captions only, and pad/truncate to `max_len=32`.

In [ ]:
class SimpleTokenizer:
    PAD = 0
    UNK = 1

    def __init__(self, texts, max_len=32, min_freq=1):
        self.max_len = max_len
        counts = defaultdict(int)
        for text in texts:
            for tok in self.tokenize(text):
                counts[tok] += 1
        self.stoi = {"<pad>": self.PAD, "<unk>": self.UNK}
        for tok, count in sorted(counts.items()):
            if count >= min_freq and tok not in self.stoi:
                self.stoi[tok] = len(self.stoi)
        self.itos = {i: s for s, i in self.stoi.items()}

    @staticmethod
    def tokenize(text):
        text = str(text).lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        return text.strip().split()

    def encode(self, text):
        ids = [self.stoi.get(tok, self.UNK) for tok in self.tokenize(text)]
        ids = ids[:self.max_len]
        ids += [self.PAD] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

TOKENIZER = SimpleTokenizer(train_df["caption"].tolist(), max_len=CFG.MAX_LEN)
print("Vocabulary size:", len(TOKENIZER.stoi))
print("Example:", train_df.iloc[0]["caption"])
print("Tokens:", TOKENIZER.encode(train_df.iloc[0]["caption"])[:12].tolist())

## 6. Flickr8k Dataset and DataLoaders

Each row is one positive `(image, caption)` pair. Since each image has multiple captions, the same image can appear in multiple rows with different captions. Evaluation later groups positives by `image_id`.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class Flickr8kDataset(Dataset):
    def __init__(self, dataframe, tokenizer=None, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        caption = str(row["caption"])
        image_id = str(row["image_id"])
        item = {"image": image, "caption": caption, "image_id": image_id}
        if self.tokenizer is not None:
            item["tokens"] = self.tokenizer.encode(caption)
        return item


def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),
        "tokens": torch.stack([x["tokens"] for x in batch]),
        "caption": [x["caption"] for x in batch],
        "image_id": [x["image_id"] for x in batch],
    }


def make_loader(df, transform, shuffle=False, batch_size=None):
    return DataLoader(
        Flickr8kDataset(df, tokenizer=TOKENIZER, transform=transform),
        batch_size=batch_size or CFG.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collate_fn,
        drop_last=False,
    )

train_loader = make_loader(train_df, train_transform, shuffle=True)
val_loader = make_loader(val_df, eval_transform, shuffle=False)
test_loader = make_loader(test_df, eval_transform, shuffle=False)

batch = next(iter(train_loader))
print(batch["image"].shape, batch["tokens"].shape, batch["image_id"][0], batch["caption"][0])

## 7. Model: Image Encoder, Text Encoder, Joint Embedding

The image encoder uses `torchvision.models.resnet18(weights=None)`, so it trains from scratch. The text encoder is an embedding layer plus GRU. Both projections are L2-normalized into the same `EMBED_DIM` space.

In [ ]:
def make_resnet18_no_pretrain():
    try:
        return resnet18(weights=None)
    except TypeError:
        return resnet18(pretrained=False)


def looks_like_cuda_runtime_error(exc):
    msg = str(exc).lower()
    return any(key in msg for key in [
        "cuda",
        "cudnn",
        "no kernel image is available for execution on the device",
        "invalid device function",
    ])

class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        backbone = make_resnet18_no_pretrain()
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(in_features, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, images):
        features = self.backbone(images)
        return self.proj(features)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, token_dim=128, hidden_dim=256, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, token_dim, padding_idx=pad_id)
        self.gru = nn.GRU(token_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, tokens):
        mask = tokens.ne(self.pad_id)
        x = self.embedding(tokens)
        h, _ = self.gru(x)
        h = h * mask.unsqueeze(-1)
        pooled = h.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1)
        return self.proj(pooled)

class ImageTextContrastiveModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim)
        self.text_encoder = TextEncoder(vocab_size, embed_dim)

    def encode_image(self, images):
        return F.normalize(self.image_encoder(images), dim=-1)

    def encode_text(self, tokens):
        return F.normalize(self.text_encoder(tokens), dim=-1)

    def forward(self, images, tokens):
        image_emb = self.encode_image(images)
        text_emb = self.encode_text(tokens)
        return image_emb, text_emb


def build_model_with_cuda_smoke_test():
    global DEVICE
    model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
    if DEVICE.type == "cuda":
        try:
            model.eval()
            with torch.no_grad():
                _ = model(batch["image"][:2].to(DEVICE), batch["tokens"][:2].to(DEVICE))
                torch.cuda.synchronize()
            print("CUDA ResNet18 smoke test passed.")
        except RuntimeError as exc:
            if looks_like_cuda_runtime_error(exc):
                print("CUDA model smoke test failed, rebuilding model on CPU.")
                print("RuntimeError:", str(exc)[:500])
                DEVICE = torch.device("cpu")
                model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
            else:
                raise
    return model

model = build_model_with_cuda_smoke_test()
print("Device after model smoke test:", DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 8. Symmetric InfoNCE Loss

The diagonal of the batch similarity matrix contains matching image-caption pairs. The loss averages image-to-text and text-to-image cross entropy.

In [ ]:
def symmetric_infonce_loss(image_emb, text_emb, temperature=0.07):
    logits = image_emb @ text_emb.T / temperature
    labels = torch.arange(image_emb.size(0), device=image_emb.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2.0

with torch.no_grad():
    images = batch["image"].to(DEVICE)
    tokens = batch["tokens"].to(DEVICE)
    image_emb, text_emb = model(images, tokens)
    test_loss = symmetric_infonce_loss(image_emb, text_emb, CFG.TEMPERATURE)
print("Sanity loss:", float(test_loss.detach().cpu()))

## 9. Training From Scratch

This trains for 5 epochs on Flickr8k image-caption pairs and saves checkpoint output plus training history.

In [ ]:
def move_batch(batch, device):
    return {
        "image": batch["image"].to(device, non_blocking=True),
        "tokens": batch["tokens"].to(device, non_blocking=True),
        "caption": batch["caption"],
        "image_id": batch["image_id"],
    }


def train_one_epoch(model, loader, optimizer, temperature, device, max_batches=None):
    model.train()
    running_loss = 0.0
    n_batches = 0
    t0 = time.time()
    for step, batch in enumerate(loader, start=1):
        if max_batches is not None and step > max_batches:
            break
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        image_emb, text_emb = model(batch["image"], batch["tokens"])
        loss = symmetric_infonce_loss(image_emb, text_emb, temperature)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += float(loss.item())
        n_batches += 1
        if step % 100 == 0:
            print(f"  batch {step:04d}/{len(loader)} | loss={running_loss / n_batches:.4f}")
    return running_loss / max(n_batches, 1), round(time.time() - t0, 2), n_batches

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
history = []

print("Training: Simplified ImageBind-style image-text contrastive learning on Flickr8k")
for epoch in range(1, CFG.EPOCHS + 1):
    train_loss, seconds, batches_seen = train_one_epoch(model, train_loader, optimizer, CFG.TEMPERATURE, DEVICE)
    row = {"epoch": epoch, "train_loss": train_loss, "seconds": seconds, "batches": batches_seen, "temperature": CFG.TEMPERATURE}
    history.append(row)
    print(f"Epoch {epoch}/{CFG.EPOCHS} | train_loss={train_loss:.4f} | batches={batches_seen} | time={seconds}s")

ckpt_path = CHECKPOINT_DIR / "flickr8k_image_text_epoch1.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": asdict(CFG),
    "tokenizer_stoi": TOKENIZER.stoi,
    "epoch": CFG.EPOCHS,
    "history": history,
}, ckpt_path)

history_path = OUTPUT_DIR / "flickr8k_train_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print("Saved checkpoint:", ckpt_path)
print("Saved history:", history_path)

## 10. Retrieval Evaluation

Evaluation extracts embeddings for the test split. Because each image has multiple captions, image-to-text retrieval treats any caption with the same `image_id` as a positive. Text-to-image retrieval treats the matching image filename as the positive.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    image_chunks, text_chunks = [], []
    image_ids, captions = [], []
    for batch in loader:
        batch_dev = move_batch(batch, device)
        image_emb, text_emb = model(batch_dev["image"], batch_dev["tokens"])
        image_chunks.append(image_emb.cpu())
        text_chunks.append(text_emb.cpu())
        image_ids.extend(batch["image_id"])
        captions.extend(batch["caption"])
    return {
        "image_emb_rows": torch.cat(image_chunks, dim=0),
        "text_emb": torch.cat(text_chunks, dim=0),
        "image_ids": image_ids,
        "captions": captions,
    }


def unique_image_embeddings(row_image_emb, image_ids):
    first_index = {}
    unique_ids = []
    for idx, image_id in enumerate(image_ids):
        if image_id not in first_index:
            first_index[image_id] = idx
            unique_ids.append(image_id)
    indices = torch.tensor([first_index[x] for x in unique_ids], dtype=torch.long)
    return row_image_emb[indices], unique_ids


def recall_at_k_grouped(similarity, positive_sets, ks=(1, 5, 10)):
    recalls = {}
    max_k = min(max(ks), similarity.size(1))
    ranked = similarity.topk(max_k, dim=1).indices.cpu().tolist()
    for k in ks:
        k_eff = min(k, similarity.size(1))
        hits = []
        for row_idx, positives in enumerate(positive_sets):
            topk = set(ranked[row_idx][:k_eff])
            hits.append(len(topk.intersection(positives)) > 0)
        recalls[f"R@{k}"] = float(np.mean(hits)) if hits else 0.0
    return recalls


def compute_retrieval_metrics(extracted):
    text_emb = extracted["text_emb"]
    image_emb_unique, unique_ids = unique_image_embeddings(extracted["image_emb_rows"], extracted["image_ids"])
    image_to_index = {image_id: idx for idx, image_id in enumerate(unique_ids)}

    caption_indices_by_image = defaultdict(set)
    for idx, image_id in enumerate(extracted["image_ids"]):
        caption_indices_by_image[image_id].add(idx)

    sim_i2t = image_emb_unique @ text_emb.T
    i2t_positive_sets = [caption_indices_by_image[image_id] for image_id in unique_ids]
    i2t = recall_at_k_grouped(sim_i2t, i2t_positive_sets, ks=(1, 5, 10))

    sim_t2i = text_emb @ image_emb_unique.T
    t2i_positive_sets = [{image_to_index[image_id]} for image_id in extracted["image_ids"]]
    t2i = recall_at_k_grouped(sim_t2i, t2i_positive_sets, ks=(1, 5, 10))

    return {
        "image_to_text": i2t,
        "text_to_image": t2i,
        "num_test_caption_pairs": len(extracted["captions"]),
        "num_test_images": len(unique_ids),
    }, image_emb_unique, unique_ids

extracted = extract_embeddings(model, test_loader, DEVICE)
eval_results, test_image_emb_unique, test_unique_ids = compute_retrieval_metrics(extracted)
eval_results["dataset"] = "Flickr8k"
eval_results["model"] = "Simplified ImageBind-style image-text contrastive learning on Flickr8k"
eval_results["epochs"] = CFG.EPOCHS
eval_results["temperature"] = CFG.TEMPERATURE

eval_path = OUTPUT_DIR / "flickr8k_eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)
print(json.dumps(eval_results, indent=2))
print("Saved evaluation:", eval_path)

## 11. Visualization

This section saves a learning curve and five retrieval examples from the test set. Each example shows the query image and top retrieved captions.

In [ ]:
def plot_learning_curve(history, output_path):
    epochs = [h["epoch"] for h in history]
    losses = [h["train_loss"] for h in history]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, losses, marker="o")
    ax.set_title("Flickr8k Image-Text Contrastive Training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train loss")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()


def save_retrieval_examples(extracted, image_emb_unique, unique_ids, image_dir, output_path, n_examples=5, top_k=3):
    text_emb = extracted["text_emb"]
    sim = image_emb_unique @ text_emb.T
    id_to_path = {row.image_id: row.image_path for row in test_df.drop_duplicates("image_id").itertuples()}
    n_examples = min(n_examples, len(unique_ids))
    chosen = list(range(n_examples))

    fig, axes = plt.subplots(n_examples, 2, figsize=(12, 3.0 * n_examples))
    if n_examples == 1:
        axes = np.array([axes])

    for row_pos, image_idx in enumerate(chosen):
        image_id = unique_ids[image_idx]
        pil_img = Image.open(id_to_path[image_id]).convert("RGB")
        axes[row_pos, 0].imshow(pil_img)
        axes[row_pos, 0].set_title(image_id)
        axes[row_pos, 0].axis("off")

        top_idx = sim[image_idx].topk(min(top_k, sim.size(1))).indices.cpu().tolist()
        lines = []
        for rank, cap_idx in enumerate(top_idx, start=1):
            cap = extracted["captions"][cap_idx]
            cap_image = extracted["image_ids"][cap_idx]
            marker = "*" if cap_image == image_id else " "
            lines.append(f"{rank}. [{marker}] {cap}")
        text = "\n\n".join(textwrap.fill(x, width=80) for x in lines)
        axes[row_pos, 1].text(0.0, 0.95, text, va="top", fontsize=10)
        axes[row_pos, 1].axis("off")

    fig.suptitle("Image-to-Text Retrieval Examples", fontsize=14)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()

learning_curve_path = OUTPUT_DIR / "flickr8k_learning_curve.png"
retrieval_examples_path = OUTPUT_DIR / "flickr8k_retrieval_examples.png"
plot_learning_curve(history, learning_curve_path)
save_retrieval_examples(extracted, test_image_emb_unique, test_unique_ids, IMAGES_DIR, retrieval_examples_path)
print("Saved learning curve:", learning_curve_path)
print("Saved retrieval examples:", retrieval_examples_path)

## 12. Optional Mini Ablation: Temperature

This optional cell compares `TEMPERATURE = [0.05, 0.07, 0.1]`. Each setting trains from scratch for one quick epoch capped at 100 batches, then evaluates retrieval on the test split.

In [ ]:
RUN_TEMPERATURE_ABLATION = True
TEMPERATURE_VALUES = [0.05, 0.07, 0.1]


def run_temperature_ablation():
    results = []
    for temp in TEMPERATURE_VALUES:
        print(f"\nAblation temperature={temp}")
        ab_model = ImageTextContrastiveModel(len(TOKENIZER.stoi), CFG.EMBED_DIM).to(DEVICE)
        ab_optimizer = torch.optim.AdamW(ab_model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
        loss, seconds, batches_seen = train_one_epoch(
            ab_model,
            train_loader,
            ab_optimizer,
            temperature=temp,
            device=DEVICE,
            max_batches=CFG.ABLATION_MAX_BATCHES,
        )
        ab_extracted = extract_embeddings(ab_model, test_loader, DEVICE)
        ab_eval, _, _ = compute_retrieval_metrics(ab_extracted)
        row = {
            "temperature": temp,
            "train_loss": loss,
            "seconds": seconds,
            "batches": batches_seen,
            "image_to_text": ab_eval["image_to_text"],
            "text_to_image": ab_eval["text_to_image"],
        }
        results.append(row)
        print(row)
    return results

if RUN_TEMPERATURE_ABLATION:
    ablation_results = run_temperature_ablation()
else:
    ablation_results = []

ablation_path = OUTPUT_DIR / "flickr8k_ablation_temperature.json"
with open(ablation_path, "w") as f:
    json.dump(ablation_results, f, indent=2)
print("Saved temperature ablation:", ablation_path)

## 13. Final Summary

The final table reports the dataset, modality, pair type, epochs, final loss, and both retrieval directions.

In [ ]:
final_loss = history[-1]["train_loss"] if history else None
summary = pd.DataFrame([
    {
        "Dataset": "Flickr8k",
        "Modality": "Image + Text",
        "Pair type": "image-caption positive pairs",
        "Epochs": CFG.EPOCHS,
        "Loss": final_loss,
        "I2T R@1": eval_results["image_to_text"]["R@1"],
        "I2T R@5": eval_results["image_to_text"]["R@5"],
        "I2T R@10": eval_results["image_to_text"]["R@10"],
        "T2I R@1": eval_results["text_to_image"]["R@1"],
        "T2I R@5": eval_results["text_to_image"]["R@5"],
        "T2I R@10": eval_results["text_to_image"]["R@10"],
    }
])
print("Simplified ImageBind-style image-text contrastive learning on Flickr8k.")
display(summary)

summary_path = OUTPUT_DIR / "flickr8k_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary.to_dict(orient="records"), f, indent=2)
print("Saved summary:", summary_path)

## ADDON: Rubric completion and missing components

This addon preserves all original cells and only standardizes missing rubric deliverables at the end of the notebook.

It writes new files to /kaggle/working/imagebind_rubric_outputs/02_flickr8k_text/ on Kaggle, with a local ./kaggle_working fallback. Optional visualizations and ablations are wrapped in try/except; if the needed runtime variables are unavailable, the addon writes NOT_RUN or FAILED_WITH_REASON instead of inventing results.

Flickr8k addon standardizes retrieval metrics, temperature ablation outputs, recall bar, learning curve, retrieval example status, and rubric report.

In [ ]:
from pathlib import Path
import json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
ADDON_DIR = ROOT_OUT / "imagebind_rubric_outputs" / "02_flickr8k_text"
ADDON_DIR.mkdir(parents=True, exist_ok=True)
FAST_ABLATION = True

def _jsonable(x):
    if isinstance(x, dict): return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [_jsonable(v) for v in x]
    if hasattr(x, "tolist"): return x.tolist()
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    return x

def _load_json_candidates(names):
    roots = [ROOT_OUT, Path.cwd(), Path("ResultForText2")]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                with open(p, "r", encoding="utf-8") as f:
                    return json.load(f), str(p)
    return None, None

eval_json, eval_source = _load_json_candidates(["flickr8k_eval_results.json"])
ab_json, ab_source = _load_json_candidates(["flickr8k_ablation_temperature.json", "flickr8k_ablation_summary.json"])
history_json, history_source = _load_json_candidates(["flickr8k_train_history.json"])

runtime_eval = globals().get("eval_results", None)
runtime_ab = globals().get("ablation_results", None)
runtime_history = globals().get("history", None)
metrics = runtime_eval if isinstance(runtime_eval, dict) else (eval_json or {})

eval_summary = {
    "dataset": "Flickr8k",
    "modality": "Image-Text",
    "model_name": "ImageTextContrastiveModel",
    "train_from_scratch": True,
    "pretrained_weights": False,
    "epochs": getattr(globals().get("CFG", None), "EPOCHS", metrics.get("epochs", "UNKNOWN")),
    "loss": "symmetric InfoNCE / contrastive loss",
    "main_metrics": {k: float(v) for k, v in metrics.items() if ("R@" in str(k) or "recall" in str(k).lower()) and isinstance(v, (int, float))},
    "ablation_status": "PRESENT_OR_STANDARDIZED",
    "limitation": "Baseline run uses 5 epochs; convergence trend is still lightweight because scratch encoders remain small.",
    "sources": {"eval": eval_source, "ablation": ab_source, "history": history_source},
}
with open(ADDON_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(eval_summary), f, indent=2)

ablation_summary = runtime_ab if isinstance(runtime_ab, list) else (ab_json or [{"status": "NOT_RUN", "reason": "No ablation results found."}])
with open(ADDON_DIR / "flickr8k_ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(ablation_summary), f, indent=2)
with open(ADDON_DIR / "ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(ablation_summary), f, indent=2)

try:
    df_ab = pd.DataFrame(ablation_summary)
    if "temperature" in df_ab.columns:
        fig, ax = plt.subplots(figsize=(7, 4))
        for col in df_ab.columns:
            if "R@1" in col or "r@1" in col.lower():
                ax.plot(df_ab["temperature"], df_ab[col], marker="o", label=col)
        ax.set_xlabel("Temperature"); ax.set_ylabel("Recall@1"); ax.set_title("Flickr8k Temperature Ablation")
        ax.legend(); fig.tight_layout()
        fig.savefig(ADDON_DIR / "ablation_summary.png", dpi=160)
        plt.show()
except Exception as exc:
    print("Ablation plot skipped:", exc)

try:
    keys = ["image_to_text_R@1", "image_to_text_R@5", "image_to_text_R@10", "text_to_image_R@1", "text_to_image_R@5", "text_to_image_R@10"]
    alt_keys = ["i2t_R@1", "i2t_R@5", "i2t_R@10", "t2i_R@1", "t2i_R@5", "t2i_R@10"]
    vals = [metrics.get(k, metrics.get(ak, np.nan)) for k, ak in zip(keys, alt_keys)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(["I2T R@1","I2T R@5","I2T R@10","T2I R@1","T2I R@5","T2I R@10"], vals)
    ax.set_ylim(0, 1); ax.set_ylabel("Recall"); ax.set_title("Flickr8k Retrieval Recall")
    ax.tick_params(axis="x", rotation=25); fig.tight_layout()
    fig.savefig(ADDON_DIR / "flickr8k_recall_bar.png", dpi=160)
    fig.savefig(ADDON_DIR / "retrieval_bar.png", dpi=160)
    plt.show()
except Exception as exc:
    print("Recall bar skipped:", exc)

try:
    hist = runtime_history if isinstance(runtime_history, list) else history_json
    if hist:
        df = pd.DataFrame(hist)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(df["epoch"], df["train_loss"], marker="o", label="train_loss")
        ax.set_title("Flickr8k Learning Curve")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
        fig.tight_layout(); fig.savefig(ADDON_DIR / "learning_curve.png", dpi=160); plt.show()
except Exception as exc:
    print("Learning curve skipped:", exc)

examples_status = {"status": "PRESENT" if (ROOT_OUT / "flickr8k_retrieval_examples.png").exists() or Path("ResultForText2/flickr8k_retrieval_examples.png").exists() else "NOT_RUN", "note": "Original notebook saves retrieval examples when run end-to-end."}
with open(ADDON_DIR / "retrieval_examples.txt", "w", encoding="utf-8") as f:
    f.write(json.dumps(examples_status, indent=2))

report = f"""Flickr8k rubric addon report
============================
Overview: image-text contrastive retrieval on Flickr8k.
Method: image and caption encoders are trained from scratch and projected into a shared embedding space using symmetric InfoNCE. This follows the ImageBind idea at mini scale: paired modalities are pulled together and in-batch negatives are pushed apart.
Implementation: train_from_scratch=True, pretrained_weights=False, epochs={eval_summary['epochs']}.
Evaluation: {json.dumps(eval_summary['main_metrics'], indent=2)}
Ablation: standardized to flickr8k_ablation_summary.json and ablation_summary.png when temperature results are available.
Limitations: Baseline run uses 5 epochs, but scratch encoders are still expected to underperform pretrained CLIP/ImageBind.
Future work: longer training, projection head ablation, hard negatives, pretrained comparison, and stronger augmentation.
"""
with open(ADDON_DIR / "flickr8k_rubric_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("ADDON outputs:", ADDON_DIR)
print(json.dumps(eval_summary, indent=2))

## ADDON: Export standardized outputs

This cell standardizes this notebook's outputs into its required project folder. It copies available files, creates a short README/report if needed, and never crashes when optional outputs are missing.

In [ ]:
from pathlib import Path
import json
import shutil
import traceback

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
STANDARD_OUTPUT_DIR = WORKING_DIR / "ResultForText2"
STANDARD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_STEM = "02_flickr8k_text"
DATASET_NAME = "Flickr8k"
MODALITY_NAME = "Image-Text"
MAX_CHECKPOINT_BYTES = 100 * 1024 * 1024
CHECKPOINT_SUFFIXES = {".pth", ".pt", ".ckpt"}

FILE_EXPORTS = {
    "train_history.json": ["train_history.json", "flickr8k_train_history.json"],
    "eval_results.json": ["eval_results.json", "flickr8k_eval_results.json"],
    "final_eval_summary.json": ["final_eval_summary.json"],
    "retrieval_recall.json": ["retrieval_recall.json", "flickr8k_eval_results.json"],
    "ablation_summary.json": ["ablation_summary.json", "flickr8k_ablation_summary.json", "flickr8k_ablation_temperature.json"],
    "learning_curve.png": ["learning_curve.png", "flickr8k_learning_curve.png"],
    "report.txt": ["report.txt", "flickr8k_bao_cao_ngan_gon.txt", "flickr8k_rubric_report.txt"],
    "rubric_report.txt": ["rubric_report.txt", "flickr8k_rubric_report.txt"],
    "retrieval_examples.png": ["retrieval_examples.png", "flickr8k_retrieval_examples.png"],
    "flickr8k_recall_bar.png": ["flickr8k_recall_bar.png", "retrieval_bar.png"],
}

def _as_path(value):
    try:
        return Path(value)
    except Exception:
        return None

def _candidate_dirs():
    dirs = [
        WORKING_DIR,
        WORKING_DIR / "outputs",
        WORKING_DIR / "results",
        WORKING_DIR / "figures",
        WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM,
        Path("."),
        Path("outputs"),
        Path("results"),
        Path("figures"),
        Path("ResultForText2"),
    ]
    for var_name in ["OUTPUT_DIR", "RESULT_DIR", "FIGURE_DIR"]:
        if var_name in globals():
            p = _as_path(globals()[var_name])
            if p is not None:
                dirs.append(p)
                dirs.append(p / "results")
                dirs.append(p / "figures")
    seen = set()
    out = []
    for d in dirs:
        try:
            key = str(d.resolve()) if d.exists() else str(d)
        except Exception:
            key = str(d)
        if key not in seen:
            out.append(d)
            seen.add(key)
    return out

def _skip_reason(path):
    try:
        if path.suffix.lower() in CHECKPOINT_SUFFIXES and path.stat().st_size > MAX_CHECKPOINT_BYTES:
            return f"large checkpoint >100MB ({path.stat().st_size / (1024**2):.1f} MB)"
    except Exception as exc:
        return f"stat failed: {exc}"
    return None

def _copy_file(src, dst):
    reason = _skip_reason(src)
    if reason:
        return {"source": str(src), "dest": str(dst), "status": "SKIPPED", "reason": reason}
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            if src.resolve() == dst.resolve():
                return {"source": str(src), "dest": str(dst), "status": "ALREADY_IN_PLACE"}
        except Exception:
            pass
        shutil.copy2(src, dst)
        return {"source": str(src), "dest": str(dst), "status": "COPIED"}
    except Exception as exc:
        return {"source": str(src), "dest": str(dst), "status": "FAILED", "reason": repr(exc)}

def _find_source(names):
    for base in _candidate_dirs():
        for name in names:
            p = Path(name)
            candidates = []
            if p.is_absolute():
                candidates.append(p)
            else:
                candidates.append(base / name)
                try:
                    if base.exists():
                        candidates.extend(base.rglob(name))
                except Exception:
                    pass
            for cand in candidates:
                if cand.exists() and cand.is_file():
                    return cand
    return None

manifest = []
for dest_name, source_names in FILE_EXPORTS.items():
    src = _find_source(source_names)
    dst = STANDARD_OUTPUT_DIR / dest_name
    if src is None:
        msg = {"dest": dest_name, "status": "MISSING", "searched_names": source_names}
        manifest.append(msg)
        print("WARNING missing output for", dest_name, "searched:", source_names)
        continue
    manifest.append(_copy_file(src, dst))

# Copy any extra rubric addon files into the standardized folder without overwriting canonical names.
addon_dir = WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM
if addon_dir.exists():
    for src in sorted(addon_dir.rglob("*")):
        if not src.is_file():
            continue
        dst = STANDARD_OUTPUT_DIR / src.name
        if dst.exists():
            continue
        manifest.append(_copy_file(src, dst))

if not (STANDARD_OUTPUT_DIR / "final_eval_summary.json").exists() and not (STANDARD_OUTPUT_DIR / "eval_results.json").exists():
    fallback_summary = {
        "dataset": DATASET_NAME,
        "modality": MODALITY_NAME,
        "status": "NOT_RUN",
        "reason": "No eval_results.json or final_eval_summary.json was found by the export addon.",
    }
    with open(STANDARD_OUTPUT_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
        json.dump(fallback_summary, f, indent=2)
    manifest.append({"dest": "final_eval_summary.json", "status": "CREATED_NOT_RUN"})

if not any((STANDARD_OUTPUT_DIR / name).exists() for name in ["report.txt", "rubric_report.txt"]):
    report = f"""Standardized output report
Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Notebook: {NOTEBOOK_STEM}
Status: exported available files. Missing files are listed in export_manifest.json.
"""
    with open(STANDARD_OUTPUT_DIR / "report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    manifest.append({"dest": "report.txt", "status": "CREATED_MINIMAL"})

readme = f"""# {NOTEBOOK_STEM} standardized outputs

Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Folder: {STANDARD_OUTPUT_DIR}

This folder is created by the export addon. Missing optional files are warnings, not fatal errors.
See export_manifest.json for copied, missing, and skipped files.
"""
with open(STANDARD_OUTPUT_DIR / "README_outputs.md", "w", encoding="utf-8") as f:
    f.write(readme)

with open(STANDARD_OUTPUT_DIR / "export_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Standardized output directory:", STANDARD_OUTPUT_DIR)
print("Exported files now present:")
for p in sorted(STANDARD_OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(STANDARD_OUTPUT_DIR))

## ADDON: Excel report

This cell creates a styled Excel workbook from JSON/TXT artifacts already exported into this notebook output folder. Missing files are recorded as NOT_AVAILABLE instead of stopping the notebook.

In [ ]:
from pathlib import Path
import json

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
EXCEL_OUTPUT_DIR = WORKING_DIR / "ResultForText2"
EXCEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_REPORT_PATH = EXCEL_OUTPUT_DIR / "02_flickr8k_text_excel_report.xlsx"

METRIC_SET = "flickr8k"

METRIC_SPECS = {
    "esc50": [
        ("test_top1", ["test_top1", "top1", "top1_acc", "test_accuracy"], "Test top-1 classification accuracy"),
        ("test_top5", ["test_top5", "top5", "top5_acc"], "Test top-5 classification accuracy"),
        ("audio_to_text_R@1", ["audio_to_text_R@1", "audio_to_text_r1", "a2t_R@1"], "Audio-to-text Recall@1"),
        ("audio_to_text_R@5", ["audio_to_text_R@5", "audio_to_text_r5", "a2t_R@5"], "Audio-to-text Recall@5"),
        ("audio_to_text_R@10", ["audio_to_text_R@10", "audio_to_text_r10", "a2t_R@10"], "Audio-to-text Recall@10"),
        ("text_to_audio_R@1", ["text_to_audio_R@1", "text_to_audio_r1", "t2a_R@1"], "Text-to-audio Recall@1"),
        ("text_to_audio_R@5", ["text_to_audio_R@5", "text_to_audio_r5", "t2a_R@5"], "Text-to-audio Recall@5"),
        ("text_to_audio_R@10", ["text_to_audio_R@10", "text_to_audio_r10", "t2a_R@10"], "Text-to-audio Recall@10"),
    ],
    "flickr8k": [
        ("image_to_text_R@1", ["image_to_text_R@1", "image_to_text_r1", "i2t_R@1", "I2T R@1"], "Image-to-text Recall@1"),
        ("image_to_text_R@5", ["image_to_text_R@5", "image_to_text_r5", "i2t_R@5", "I2T R@5"], "Image-to-text Recall@5"),
        ("image_to_text_R@10", ["image_to_text_R@10", "image_to_text_r10", "i2t_R@10", "I2T R@10"], "Image-to-text Recall@10"),
        ("text_to_image_R@1", ["text_to_image_R@1", "text_to_image_r1", "t2i_R@1", "T2I R@1"], "Text-to-image Recall@1"),
        ("text_to_image_R@5", ["text_to_image_R@5", "text_to_image_r5", "t2i_R@5", "T2I R@5"], "Text-to-image Recall@5"),
        ("text_to_image_R@10", ["text_to_image_R@10", "text_to_image_r10", "t2i_R@10", "T2I R@10"], "Text-to-image Recall@10"),
    ],
    "llvip": [
        ("image_to_thermal_R@1", ["image_to_thermal_R@1", "image_to_thermal_r1", "i2t_R@1"], "Visible image-to-thermal Recall@1"),
        ("image_to_thermal_R@5", ["image_to_thermal_R@5", "image_to_thermal_r5", "i2t_R@5"], "Visible image-to-thermal Recall@5"),
        ("image_to_thermal_R@10", ["image_to_thermal_R@10", "image_to_thermal_r10", "i2t_R@10"], "Visible image-to-thermal Recall@10"),
        ("thermal_to_image_R@1", ["thermal_to_image_R@1", "thermal_to_image_r1", "t2i_R@1"], "Thermal-to-visible image Recall@1"),
        ("thermal_to_image_R@5", ["thermal_to_image_R@5", "thermal_to_image_r5", "t2i_R@5"], "Thermal-to-visible image Recall@5"),
        ("thermal_to_image_R@10", ["thermal_to_image_R@10", "thermal_to_image_r10", "t2i_R@10"], "Thermal-to-visible image Recall@10"),
        ("accuracy", ["accuracy", "test_accuracy", "top1", "top1_acc"], "Optional classification/top-1 accuracy"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 accuracy"),
    ],
    "nyu": [
        ("image_to_depth_R@1", ["image_to_depth_R@1", "image_to_depth_r1", "i2d_r@1", "I2D R@1"], "RGB image-to-depth Recall@1"),
        ("image_to_depth_R@5", ["image_to_depth_R@5", "image_to_depth_r5", "i2d_r@5", "I2D R@5"], "RGB image-to-depth Recall@5"),
        ("image_to_depth_R@10", ["image_to_depth_R@10", "image_to_depth_r10", "i2d_r@10", "I2D R@10"], "RGB image-to-depth Recall@10"),
        ("depth_to_image_R@1", ["depth_to_image_R@1", "depth_to_image_r1", "d2i_r@1", "D2I R@1"], "Depth-to-RGB image Recall@1"),
        ("depth_to_image_R@5", ["depth_to_image_R@5", "depth_to_image_r5", "d2i_r@5", "D2I R@5"], "Depth-to-RGB image Recall@5"),
        ("depth_to_image_R@10", ["depth_to_image_R@10", "depth_to_image_r10", "d2i_r@10", "D2I R@10"], "Depth-to-RGB image Recall@10"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 classification accuracy"),
        ("top5", ["top5", "top5_acc", "test_top5"], "Optional top-5 classification accuracy"),
    ],
    "uci": [
        ("accuracy", ["accuracy", "test_accuracy", "val_accuracy"], "Human activity recognition accuracy"),
        ("macro_f1", ["macro_f1", "test_macro_f1", "f1_macro"], "Macro-averaged F1 score"),
        ("per_class_accuracy", ["per_class_accuracy", "class_accuracy", "per_class_acc"], "Per-class accuracy breakdown if available"),
    ],
}

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
TITLE_FILL = PatternFill("solid", fgColor="D9EAF7")
ALT_FILL = PatternFill("solid", fgColor="F7FBFF")
WHITE_FONT = Font(name="Arial", color="FFFFFF", bold=True)
TITLE_FONT = Font(name="Arial", size=14, bold=True, color="1F4E78")
BASE_FONT = Font(name="Arial", size=10)
THIN_SIDE = Side(style="thin", color="B7B7B7")
THIN_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
STATUS_FILLS = {
    "GOOD": PatternFill("solid", fgColor="C6EFCE"),
    "PRESENT": PatternFill("solid", fgColor="C6EFCE"),
    "REVIEW": PatternFill("solid", fgColor="FCE4D6"),
    "LOW": PatternFill("solid", fgColor="FFC7CE"),
    "NOT_AVAILABLE": PatternFill("solid", fgColor="E7E6E6"),
    "WARNING": PatternFill("solid", fgColor="FCE4D6"),
}

def normalize_key(value):
    return "".join(ch.lower() for ch in str(value) if ch.isalnum())

def load_json_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as exc:
        return None, f"WARNING: failed to read {path.name}: {exc}"

def ordered_json_files():
    priority = [
        "final_eval_summary.json",
        "eval_results.json",
        "retrieval_recall.json",
        "train_history.json",
        "ablation_summary.json",
    ]
    files = []
    seen = set()
    for name in priority:
        p = EXCEL_OUTPUT_DIR / name
        if p.exists() and p.is_file():
            files.append(p)
            seen.add(p.resolve())
    for p in sorted(EXCEL_OUTPUT_DIR.rglob("*.json")):
        try:
            key = p.resolve()
        except Exception:
            key = p
        if key not in seen:
            files.append(p)
            seen.add(key)
    return files

JSON_OBJECTS = []
JSON_WARNINGS = []
for json_path in ordered_json_files():
    obj, warning = load_json_file(json_path)
    if warning:
        JSON_WARNINGS.append(warning)
    else:
        JSON_OBJECTS.append((json_path.name, obj))

TXT_WARNINGS = []
for txt_path in sorted(list(EXCEL_OUTPUT_DIR.rglob("*.txt")) + list(EXCEL_OUTPUT_DIR.rglob("*.md"))):
    try:
        text = txt_path.read_text(encoding="utf-8", errors="replace")
        if "WARNING" in text.upper() or "NOT_AVAILABLE" in text.upper() or "NOT_RUN" in text.upper():
            TXT_WARNINGS.append(f"See {txt_path.name} for warnings/status notes")
    except Exception as exc:
        TXT_WARNINGS.append(f"WARNING: failed to read {txt_path.name}: {exc}")

def flatten_json(obj, prefix="", source=""):
    rows = []
    if prefix:
        rows.append((prefix, obj, source))
    if isinstance(obj, dict):
        for key, value in obj.items():
            child = f"{prefix}.{key}" if prefix else str(key)
            rows.extend(flatten_json(value, child, source))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            child = f"{prefix}.{idx}" if prefix else str(idx)
            rows.extend(flatten_json(value, child, source))
    return rows

FLAT_JSON = []
for source, obj in JSON_OBJECTS:
    FLAT_JSON.extend(flatten_json(obj, "", source))

def find_value(candidates):
    normalized_candidates = [normalize_key(c) for c in candidates]
    for cand in normalized_candidates:
        for key, value, source in FLAT_JSON:
            last_part = key.split(".")[-1]
            if normalize_key(last_part) == cand:
                return value, f"source: {source} / {key}"
    for cand in normalized_candidates:
        if len(cand) < 6:
            continue
        for key, value, source in FLAT_JSON:
            if normalize_key(key).endswith(cand):
                return value, f"source: {source} / {key}"
    return None, "NOT_AVAILABLE"

def safe_float(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value)
        except Exception:
            return None
    return None

def display_value(value):
    if value is None:
        return "NOT_AVAILABLE"
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, (int, float)):
        return round(float(value), 6)
    if isinstance(value, (dict, list)):
        rendered = json.dumps(value, ensure_ascii=False)
        return rendered[:300] + ("..." if len(rendered) > 300 else "")
    return str(value)

def status_for(metric, value):
    if value is None or value == "NOT_AVAILABLE":
        return "NOT_AVAILABLE"
    if isinstance(value, (dict, list)):
        return "PRESENT" if value else "NOT_AVAILABLE"
    score = safe_float(value)
    if score is None:
        return "PRESENT"
    if score > 1.0 and score <= 100.0:
        score = score / 100.0
    metric_l = str(metric).lower()
    if "loss" in metric_l:
        if score <= 0.5:
            return "GOOD"
        if score <= 1.5:
            return "REVIEW"
        return "LOW"
    if score >= 0.70:
        return "GOOD"
    if score >= 0.40:
        return "REVIEW"
    return "LOW"

def get_by_alias(row, aliases):
    if not isinstance(row, dict):
        return None
    lookup = {normalize_key(k): v for k, v in row.items()}
    for alias in aliases:
        key = normalize_key(alias)
        if key in lookup:
            return lookup[key]
    return None

def find_train_history():
    candidates = [EXCEL_OUTPUT_DIR / "train_history.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*train*history*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        if isinstance(obj, list):
            return obj, path.name
        if isinstance(obj, dict):
            for key in ["history", "train_history", "rows"]:
                if isinstance(obj.get(key), list):
                    return obj[key], path.name
    return None, None

def build_training_rows():
    history, source = find_train_history()
    if not history:
        return ["Status", "Notes"], [["NOT_AVAILABLE", "train_history.json not found in output folder"]]
    alias_map = {
        "epoch": ["epoch"],
        "train_loss": ["train_loss", "loss_train", "training_loss"],
        "val_loss": ["val_loss", "test_loss", "valid_loss", "validation_loss"],
        "train_top1": ["train_top1", "train_accuracy", "train_acc", "train_i2t_acc", "train_top1_acc"],
        "val_top1": ["val_top1", "val_accuracy", "val_acc", "test_accuracy", "val_i2t_acc", "test_top1"],
        "lr": ["lr", "learning_rate"],
    }
    normalized_rows = []
    for raw in history:
        if not isinstance(raw, dict):
            continue
        row = {col: get_by_alias(raw, aliases) for col, aliases in alias_map.items()}
        normalized_rows.append(row)
    columns = [col for col in ["epoch", "train_loss", "val_loss", "train_top1", "val_top1", "lr"] if any(row.get(col) is not None for row in normalized_rows)]
    if "epoch" not in columns:
        columns = ["epoch"] + columns
    rows = [[display_value(row.get(col)) for col in columns] for row in normalized_rows]
    best_row = None
    best_note = None
    val_top1_values = [(safe_float(row.get("val_top1")), idx) for idx, row in enumerate(normalized_rows)]
    val_top1_values = [(v, idx) for v, idx in val_top1_values if v is not None]
    if val_top1_values:
        _, idx = max(val_top1_values, key=lambda x: x[0])
        best_row = normalized_rows[idx]
        best_note = "Best Epoch by val_top1"
    else:
        val_loss_values = [(safe_float(row.get("val_loss")), idx) for idx, row in enumerate(normalized_rows)]
        val_loss_values = [(v, idx) for v, idx in val_loss_values if v is not None]
        if val_loss_values:
            _, idx = min(val_loss_values, key=lambda x: x[0])
            best_row = normalized_rows[idx]
            best_note = "Best Epoch by val_loss"
    if best_row:
        best = [display_value(best_row.get(col)) for col in columns]
        if best:
            best[0] = f"Best Epoch: {best[0]}"
        rows.append(best)
        rows.append([best_note] + [""] * (len(columns) - 1))
    rows.append([f"source: {source}"] + [""] * (len(columns) - 1))
    return columns, rows

def find_ablation_object():
    candidates = [EXCEL_OUTPUT_DIR / "ablation_summary.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*ablation*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        return obj, path.name
    return None, None

def setting_name(record, index):
    if not isinstance(record, dict):
        return f"setting_{index}"
    for key in ["setting", "name", "temperature", "lr", "learning_rate", "hidden_dim", "batch_size"]:
        if key in record:
            return f"{key}={record[key]}" if key not in ["setting", "name"] else str(record[key])
    return f"setting_{index}"

def build_ablation_rows():
    obj, source = find_ablation_object()
    if obj is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "No ablation JSON found in output folder"]]
    records = obj if isinstance(obj, list) else obj.get("results", obj.get("ablation", [obj])) if isinstance(obj, dict) else []
    if isinstance(records, dict):
        records = [records]
    rows = []
    for idx, record in enumerate(records):
        if not isinstance(record, dict):
            rows.append([f"setting_{idx}", "value", display_value(record), f"source: {source}"])
            continue
        flat = flatten_json(record, "", source)
        numeric_items = []
        for key, value, _ in flat:
            if safe_float(value) is not None and key.split(".")[-1].lower() not in ["epoch", "epochs", "seed"]:
                numeric_items.append((key, value))
        if numeric_items:
            for key, value in numeric_items[:12]:
                rows.append([setting_name(record, idx), key, display_value(value), f"source: {source}"])
        else:
            note = record.get("reason", record.get("note", "No numeric metric found"))
            status = record.get("status", "NOT_AVAILABLE")
            rows.append([setting_name(record, idx), "status", status, f"{note}; source: {source}"])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", f"No ablation records in {source}"]]

def find_per_class():
    for key, value, source in FLAT_JSON:
        if normalize_key(key.split(".")[-1]) in ["perclassaccuracy", "classaccuracy", "perclassacc"]:
            return value, source, key
    return None, None, None

def build_per_class_rows():
    value, source, key = find_per_class()
    if value is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE"]]
    rows = []
    if isinstance(value, dict):
        for cls, acc in value.items():
            rows.append([str(cls), display_value(acc), status_for("accuracy", acc)])
    elif isinstance(value, list):
        for idx, acc in enumerate(value):
            rows.append([str(idx), display_value(acc), status_for("accuracy", acc)])
    else:
        rows.append(["per_class_accuracy", display_value(value), status_for("accuracy", value)])
    rows.append([f"source: {source}", key, ""])
    return rows

def purpose_for(path):
    suffix = path.suffix.lower()
    if suffix == ".json":
        return "metrics/data"
    if suffix == ".png":
        return "figure/chart"
    if suffix in [".txt", ".md"]:
        return "report"
    if suffix == ".xlsx":
        return "excel report"
    if suffix == ".zip":
        return "compressed output"
    if suffix in [".pth", ".pt", ".ckpt"]:
        return "checkpoint"
    return "output artifact"

def build_manifest_rows():
    rows = []
    for path in sorted(EXCEL_OUTPUT_DIR.rglob("*")):
        if not path.is_file():
            continue
        try:
            rel = str(path.relative_to(EXCEL_OUTPUT_DIR))
        except Exception:
            rel = path.name
        try:
            size_kb = round(path.stat().st_size / 1024, 2)
        except Exception:
            size_kb = "NOT_AVAILABLE"
        rows.append([rel, path.suffix.lower() or "NO_EXTENSION", size_kb, purpose_for(path)])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "Output folder is empty"]]

def autosize(ws):
    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = 0
        for cell in column_cells:
            try:
                max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 55)

def create_table_sheet(wb, title, headers, rows):
    ws = wb.create_sheet(title)
    ws.append([title] + [""] * (len(headers) - 1))
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(headers))
    title_cell = ws.cell(row=1, column=1)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    title_cell.alignment = Alignment(horizontal="center")
    ws.append([""] * len(headers))
    ws.append(headers)
    for row in rows:
        fixed = list(row)[:len(headers)] + [""] * max(0, len(headers) - len(row))
        ws.append(fixed)
    for row in ws.iter_rows():
        for cell in row:
            cell.font = BASE_FONT
            cell.border = THIN_BORDER
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    for cell in ws[3]:
        cell.fill = HEADER_FILL
        cell.font = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    status_cols = [idx + 1 for idx, name in enumerate(headers) if "status" in name.lower()]
    for row_idx in range(4, ws.max_row + 1):
        if row_idx % 2 == 0:
            for col_idx in range(1, len(headers) + 1):
                ws.cell(row=row_idx, column=col_idx).fill = ALT_FILL
        for col_idx in status_cols:
            value = str(ws.cell(row=row_idx, column=col_idx).value or "")
            fill = STATUS_FILLS.get(value)
            if fill:
                ws.cell(row=row_idx, column=col_idx).fill = fill
    ws.freeze_panes = "A4"
    autosize(ws)
    return ws

metric_rows = []
for metric, candidates, meaning in METRIC_SPECS[METRIC_SET]:
    value, note = find_value([metric] + candidates)
    metric_rows.append([metric, display_value(value), meaning, status_for(metric, value), note])
if JSON_WARNINGS or TXT_WARNINGS:
    for warning in JSON_WARNINGS[:5] + TXT_WARNINGS[:5]:
        metric_rows.append(["warning", warning, "Runtime/file warning", "WARNING", "Read from JSON/TXT files only"])

training_headers, training_rows = build_training_rows()
ablation_rows = build_ablation_rows()
per_class_rows = build_per_class_rows()

wb = Workbook()
wb.remove(wb.active)
create_table_sheet(wb, "Evaluation Metrics", ["Metric", "Value", "Benchmark/Meaning", "Status", "Notes"], metric_rows)
create_table_sheet(wb, "Training Summary", training_headers, training_rows)
create_table_sheet(wb, "Ablation Summary", ["Setting", "Metric", "Value", "Notes"], ablation_rows)
per_class_ws = create_table_sheet(wb, "Per-Class Breakdown", ["Class", "Accuracy", "Status"], per_class_rows)
per_class_ws.cell(row=1, column=1).value = "Per-Class / Breakdown"

# Save once so the manifest can include the Excel report itself.
wb.save(EXCEL_REPORT_PATH)
manifest_rows = build_manifest_rows()
create_table_sheet(wb, "Output Manifest", ["File name", "Extension", "Size KB", "Purpose"], manifest_rows)
wb.save(EXCEL_REPORT_PATH)

print(f"Excel report saved to: {EXCEL_REPORT_PATH}")
print(f"Total sheets: {len(wb.sheetnames)}")
print(f"Total output files listed: {len(manifest_rows)}")

## ADDON: Zip this notebook output

This cell zips this notebook's standardized output folder for direct Kaggle download. It skips missing folders safely and excludes large checkpoints over 100MB.

In [ ]:
import os
import zipfile
from pathlib import Path

def zip_folder(output_dir, zip_path, max_checkpoint_mb=100):
    output_dir = Path(output_dir)
    zip_path = Path(zip_path)

    if not output_dir.exists():
        print(f"WARNING: output folder does not exist: {output_dir}")
        return None

    added_files = []
    skipped_files = []

    def should_skip(file_path):
        suffix = file_path.suffix.lower()
        if suffix in [".pth", ".pt", ".ckpt"]:
            size_mb = file_path.stat().st_size / (1024 * 1024)
            if size_mb > max_checkpoint_mb:
                return True
        return False

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_dir.rglob("*"):
            if not file_path.is_file():
                continue

            if should_skip(file_path):
                skipped_files.append(str(file_path))
                continue

            arcname = file_path.relative_to(output_dir.parent)
            zipf.write(file_path, arcname)
            added_files.append(str(arcname))

    size_mb = zip_path.stat().st_size / (1024 * 1024)

    print("=" * 80)
    print(f"ZIP CREATED: {zip_path}")
    print(f"ZIP SIZE: {size_mb:.2f} MB")
    print(f"TOTAL FILES ADDED: {len(added_files)}")
    print("=" * 80)

    for f in added_files:
        print(f"- {f}")

    if skipped_files:
        print("=" * 80)
        print("SKIPPED LARGE CHECKPOINTS:")
        for f in skipped_files:
            print(f"- {f}")

    print("=" * 80)
    print("Download path:")
    print(str(zip_path))

    return zip_path

zip_folder(
    "/kaggle/working/ResultForText2",
    "/kaggle/working/ResultForText2.zip"
)